<a target="_blank" href="https://colab.research.google.com/github/TransformerLensOrg/TransformerLens/blob/main/demos/Grokking_Demo.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

> **📋 Diff from original:** The original notebook's cell 1 is just:
> ```
> # Grokking Demo Notebook
> # WARNING: This notebook is designed to run on a GPU ...
> ```
> This version adds a full description of the algorithm, grokking, and the notebook's structure.


# Grokking Demo Notebook

This notebook accompanies the paper **"Progress measures for grokking via mechanistic interpretability"** (Nanda et al., 2023, [arXiv:2301.05217](https://arxiv.org/abs/2301.05217)).

**What is grokking?** A neural network first *memorizes* the training data (train loss drops, test loss stays high), then much later suddenly *generalizes* (test loss drops). This delayed generalization is called **grokking**.

**What this notebook does:**
1. Trains a **1-layer transformer** on **modular addition**: given inputs `a` and `b`, predict `(a + b) mod 113`
2. Observes grokking — the model memorizes the training set within ~1,500 epochs but doesn't generalize until ~13,000 epochs
3. **Reverse-engineers the algorithm** the model learns: it uses the **Discrete Fourier Transform** and **trigonometric identities** to convert addition into rotation on a circle
4. Defines **progress measures** (restricted loss, excluded loss) that reveal grokking is a gradual 3-phase process (memorization → circuit formation → cleanup), not a sudden transition

**The algorithm the model learns:**
- **Embed**: Map token `a` to `sin(wₖa)`, `cos(wₖa)` for 5 key frequencies wₖ = 2πk/113
- **Attend**: Copy embeddings of `a` and `b` to the `=` position
- **MLP**: Compute `cos(wₖ(a+b))` using the trig identity `cos(α+β) = cos(α)cos(β) − sin(α)sin(β)`
- **Unembed**: Produce logits ∝ `Σₖ cos(wₖ(a+b−c))`, which peak at `c = (a+b) mod 113` via constructive interference

<b style="color: red">To use this notebook, go to Runtime > Change Runtime Type and select GPU as the hardware accelerator.</b>

# Setup
(No need to read)

In [1]:
# Boolean flag: True = train a new model from scratch (~2 min on GPU)
# False = load a previously saved checkpoint from disk (skips training)
TRAIN_MODEL = True

`TRAIN_MODEL = True` trains from scratch (~2 min on GPU). Set to `False` to load a previously saved checkpoint.

**Environment setup**: The cell below detects whether we're running in Google Colab or a local Jupyter notebook. In local mode, it enables `autoreload` so edits to TransformerLens source code take effect without restarting the kernel. In Colab/CI, it installs the required packages.

> **📋 Diff from original:** The original uses the deprecated `ipython.magic("load_ext autoreload")` and `ipython.magic("autoreload 2")`. This version uses the modern `ipython.run_line_magic("load_ext", "autoreload")` API.


In [2]:
# --- Environment Detection ---
# Detect whether we're running in Google Colab, GitHub Actions CI, or a local Jupyter notebook.
# Each environment needs different setup (package installation, autoreload, etc.)
import os

DEVELOPMENT_MODE = True  # Flag for local development settings (affects Plotly renderer)
IN_GITHUB = os.getenv("GITHUB_ACTIONS") == "true"  # Check if running in GitHub Actions CI

try:
    import google.colab  # This import only succeeds inside Google Colab
    IN_COLAB = True
    print("Running as a Colab notebook")

    # PySvelte is an unmaintained visualization library, use it as a backup if circuitsvis isn't working
    # # Install another version of node that makes PySvelte work way faster
    # !curl -fsSL https://deb.nodesource.com/setup_16.x | sudo -E bash -; sudo apt-get install -y nodejs
    # %pip install git+https://github.com/neelnanda-io/PySvelte.git
except:
    # If google.colab import fails, we're in a local Jupyter/VSCode notebook
    IN_COLAB = False
    print("Running as a Jupyter notebook - intended for development only!")
    from IPython import get_ipython

    ipython = get_ipython()  # Get the current IPython session object
    # Enable autoreload: any changes to imported modules (e.g. TransformerLens source)
    # will be automatically reloaded before executing each cell — no kernel restart needed
    ipython.run_line_magic("load_ext", "autoreload")   # Equivalent to: %load_ext autoreload
    ipython.run_line_magic("autoreload", "2")           # Equivalent to: %autoreload 2 (reload ALL modules)

# In Colab or CI, install required packages (locally they're assumed pre-installed)
if IN_COLAB or IN_GITHUB:
    %pip install transformer_lens
    %pip install circuitsvis

Running as a Jupyter notebook - intended for development only!


In [3]:
# --- Plotly Renderer Selection ---
# Plotly needs different rendering backends depending on the environment:
# - "colab": works in Google Colab (uses Colab's built-in renderer)
# - "notebook_connected": works in local Jupyter/VSCode (uses plotly.js via CDN for interactivity)
import plotly.io as pio
if IN_COLAB or not DEVELOPMENT_MODE:
    pio.renderers.default = "colab"
else:
    pio.renderers.default = "notebook_connected"
print(f"Using renderer: {pio.renderers.default}")

Using renderer: notebook_connected


**Imports explained:**
- **`torch`** — PyTorch core (tensors, autograd, neural network modules)
- **`einops`** — Tensor reshaping with readable syntax, e.g. `rearrange(x, "(a b) c -> a b c", a=113)` reshapes a flat batch back into a 2D grid
- **`fancy_einsum`** — Named-dimension Einstein summation (more readable than `torch.einsum`)
- **`transformer_lens`** — The mechanistic interpretability library that provides `HookedTransformer` (a transformer with hooks at every intermediate activation), `ActivationCache` (stores cached activations), and `FactoredMatrix` (efficient low-rank matrix operations)

In [4]:
# --- Global Plotly Font Size Defaults ---
# Modify the default 'plotly' template so ALL subsequent figures inherit these sizes.
# This avoids repeating font size settings in every individual plot call.
pio.templates['plotly'].layout.xaxis.title.font.size = 20  # X-axis label font = 20pt
pio.templates['plotly'].layout.yaxis.title.font.size = 20  # Y-axis label font = 20pt
pio.templates['plotly'].layout.title.font.size = 30         # Plot title font = 30pt

In [5]:
# --- Main Library Imports ---
import torch                    # PyTorch core: tensors, autograd, GPU acceleration
import torch.nn as nn           # Neural network modules (Linear, Embedding, etc.)
import torch.nn.functional as F # Functional API (relu, softmax, cross_entropy, etc.)
import torch.optim as optim     # Optimizers (Adam, SGD, AdamW, etc.)
import numpy as np              # NumPy: needed for Plotly compatibility (can't plot torch tensors directly)
import einops                   # Readable tensor reshaping: rearrange("(a b) c -> a b c", a=113)
from fancy_einsum import einsum # Named-dimension Einstein summation, more readable than torch.einsum
import os                       # OS-level operations (path handling, environment variables)
import tqdm.auto as tqdm        # Progress bars (auto-detects notebook vs terminal and picks the right widget)
import random                   # Python's built-in random number generator
from pathlib import Path        # Object-oriented filesystem paths (e.g. Path("a/b").parent)
import plotly.express as px     # High-level Plotly API for quick interactive charts
from torch.utils.data import DataLoader  # (imported but not actually used in this notebook)

from typing import List, Union, Optional  # Type hint annotations
from functools import partial             # Create partially-applied functions
import copy                               # Deep copying (critical for saving model checkpoints)

import itertools                                                       # Iteration utilities (not used)
from transformers import AutoModelForCausalLM, AutoConfig, AutoTokenizer  # HuggingFace (not used)
import dataclasses                                                     # Dataclass support (not used directly)
import datasets                                                        # HuggingFace datasets (not used)
from IPython.display import HTML                                       # Render HTML in notebook output

In [6]:
# --- TransformerLens Imports ---
import transformer_lens                  # The mechanistic interpretability library
import transformer_lens.utils as utils   # Helper functions, especially utils.to_numpy() for plotting
from transformer_lens.hook_points import (
    HookedRootModule,  # Base class for models instrumented with hook points
    HookPoint,         # Individual hook point that can record/modify activations during forward pass
)
from transformer_lens import (
    HookedTransformer,        # Transformer with hooks at every intermediate computation
    HookedTransformerConfig,  # Dataclass specifying model architecture (layers, heads, dims, etc.)
    FactoredMatrix,           # Efficient low-rank matrix representation (A @ B without materializing)
    ActivationCache,          # Dict-like container storing all cached activations from run_with_cache()
)

# Use GPU if available (CUDA), otherwise fall back to CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

Plotting helper functions — thin wrappers around Plotly Express. `utils.to_numpy()` converts PyTorch tensors (potentially on GPU, with gradients attached) into numpy arrays that Plotly can consume. The `imshow` helper uses an **RdBu** (Red-Blue) colormap centered at 0, which is ideal for visualizing signed quantities like Fourier coefficients and attention weights.

In [7]:
# --- Plotting Helper Functions ---
# Thin wrappers around Plotly Express that handle PyTorch tensor → numpy conversion.

def imshow(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    """Heatmap plot. Uses RdBu (Red-Blue) diverging colormap centered at 0."""
    px.imshow(
        utils.to_numpy(tensor),              # Convert tensor (possibly on GPU, with grad) to numpy array
        color_continuous_midpoint=0.0,        # Center the colormap at zero: negative=blue, zero=white, positive=red
        color_continuous_scale="RdBu",        # Red-Blue diverging colormap — ideal for signed values
        labels={"x":xaxis, "y":yaxis},        # Axis labels
        **kwargs                              # Pass through: title, x, y tick labels, facet_col, etc.
    ).show(renderer)                          # Display the figure using the configured renderer

def line(tensor, renderer=None, xaxis="", yaxis="", **kwargs):
    """Line chart. If tensor is 2D, each row becomes a separate line."""
    px.line(
        utils.to_numpy(tensor),
        labels={"x":xaxis, "y":yaxis},
        **kwargs
    ).show(renderer)

def scatter(x, y, xaxis="", yaxis="", caxis="", renderer=None, **kwargs):
    """Scatter plot."""
    x = utils.to_numpy(x)  # Convert x coordinates from tensor to numpy
    y = utils.to_numpy(y)  # Convert y coordinates from tensor to numpy
    px.scatter(y=y, x=x, labels={"x":xaxis, "y":yaxis, "color":caxis}, **kwargs).show(renderer)

In [8]:
# --- Model Save Path ---
# Define the file path where the trained model and checkpoints will be saved/loaded.
PTH_LOCATION = "workspace/_scratch/grokking_demo.pth"

# Create the parent directory ("workspace/_scratch/") if it doesn't already exist.
# exist_ok=True prevents an error if the directory is already there.
os.makedirs(Path(PTH_LOCATION).parent, exist_ok=True)

# Model Training

## Config

Key hyperparameters:
- **`p = 113`** — A prime number used as the modulus. Primes are chosen because ℤₚ (integers mod p) forms a field with nice Fourier properties — all nonzero frequencies are "active" and the DFT is clean.
- **`frac_train = 0.3`** — Only 30% of the 113² = 12,769 possible (a,b) pairs are used for training. The model must generalize to the remaining 70%.
- **`wd = 1.0`** — **Very high weight decay**. This is critical for grokking — it penalizes large weights, slowly pushing the model from a memorization solution (large, unstructured weights) toward a compact generalizing Fourier circuit.
- **`num_epochs = 25000`** — Long enough to observe grokking (which happens around epoch 13,000).

In [9]:
# --- Hyperparameters ---
p = 113          # The prime modulus: the task is (a + b) mod 113.
                 # Primes are chosen because Z_p forms a field with clean Fourier properties.
frac_train = 0.3 # Only 30% of all 113^2 = 12,769 (a,b) pairs are used for training.
                 # The model must generalize to the remaining 70%.

# Optimizer config
lr = 1e-3        # Learning rate for AdamW optimizer
wd = 1.          # Weight decay = 1.0 — VERY high! This is critical for grokking.
                 # It penalizes large weights, slowly eroding memorization in favor of the compact Fourier circuit.
betas = (0.9, 0.98)  # Adam momentum parameters: beta1=0.9 (first moment), beta2=0.98 (second moment)

num_epochs = 25000       # Total training epochs — long enough to observe grokking (~epoch 13,000)
checkpoint_every = 100   # Save a snapshot of model weights every 100 epochs (250 checkpoints total)

DATA_SEED = 598  # Random seed for the train/test split — ensures reproducibility

## Define Task

The task is **modular addition**: given two numbers `a` and `b` (each in 0–112), predict `(a + b) mod 113`.

We construct a dataset of **all** 113² = 12,769 possible input pairs. Each input is a 3-token sequence `[a, b, =]` and the label is the correct answer.

Input format: **|a|b|=|** — three tokens at positions 0, 1, 2.

The code below creates the input vectors using `einops.repeat`:
- `a_vector`: `[0,0,...,0, 1,1,...,1, ..., 112,...,112]` — each value of `a` repeated 113 times (once per value of `b`)
- `b_vector`: `[0,1,...,112, 0,1,...,112, ...]` — cycles through all values of `b` for each `a`
- `equals_vector`: all 113s — token ID 113 represents `=` (the vocabulary is {0, 1, ..., 112, 113})

In [10]:
# --- Construct Input Vectors ---
# We need all 113^2 = 12,769 possible (a, b) pairs as 3-token sequences: [a, b, =]

# a_vector: [0,0,...,0, 1,1,...,1, ..., 112,...,112] — each value of 'a' repeated 113 times (once per 'b')
# "i -> (i j)" means: take dim i and repeat it, interleaving with a new dim j of size p
a_vector = einops.repeat(torch.arange(p), "i -> (i j)", j=p)  # shape: [12769]

# b_vector: [0,1,...,112, 0,1,...,112, ...] — cycles through all values of 'b' for each 'a'
# "j -> (i j)" tiles the range p times
b_vector = einops.repeat(torch.arange(p), "j -> (i j)", i=p)  # shape: [12769]

# equals_vector: all 113s — token ID 113 represents the '=' sign
# The vocabulary is {0, 1, ..., 112} for numbers and {113} for '='
equals_vector = einops.repeat(torch.tensor(113), " -> (i j)", i=p, j=p)  # shape: [12769]

In [11]:
# --- Build the Full Dataset Tensor ---
# Stack [a, b, =] into a single tensor where each row is one 3-token input sequence.
# torch.stack(dim=1) stacks column-wise: [12769] + [12769] + [12769] → [12769, 3]
dataset = torch.stack([a_vector, b_vector, equals_vector], dim=1).to(device)  # Move to GPU if available

print(dataset[:5])   # First 5 rows: [0,0,113], [0,1,113], [0,2,113], [0,3,113], [0,4,113]
print(dataset.shape) # torch.Size([12769, 3])

tensor([[  0,   0, 113],
        [  0,   1, 113],
        [  0,   2, 113],
        [  0,   3, 113],
        [  0,   4, 113]], device='cuda:0')
torch.Size([12769, 3])


In [12]:
# --- Compute Ground Truth Labels ---
# For each (a, b) pair, the correct answer is (a + b) mod 113.
# dataset[:, 0] = all 'a' values, dataset[:, 1] = all 'b' values
labels = (dataset[:, 0] + dataset[:, 1]) % p  # shape: [12769]

print(labels.shape)  # torch.Size([12769])
print(labels[:5])    # For a=0: (0+0)%113=0, (0+1)%113=1, (0+2)%113=2, (0+3)%113=3, (0+4)%113=4

torch.Size([12769])
tensor([0, 1, 2, 3, 4], device='cuda:0')


Random 30/70 train-test split. With only 30% of pairs for training, the model must learn the **general rule** of modular addition to perform well on the 70% it hasn't seen. This is what makes grokking interesting — the model first memorizes the 3,830 training pairs, then much later discovers the general algorithm.

In [13]:
# --- Train/Test Split ---
torch.manual_seed(DATA_SEED)       # Set seed=598 for reproducible random permutation
indices = torch.randperm(p*p)      # Random permutation of [0, 1, ..., 12768]
cutoff = int(p*p*frac_train)       # = int(12769 * 0.3) = 3830 training examples
train_indices = indices[:cutoff]   # First 3830 random indices → training set
test_indices = indices[cutoff:]    # Remaining 8939 indices → test set

# Split dataset and labels according to the random indices
train_data = dataset[train_indices]      # [3830, 3] — training input sequences
train_labels = labels[train_indices]     # [3830] — training ground truth labels
test_data = dataset[test_indices]        # [8939, 3] — test input sequences
test_labels = labels[test_indices]       # [8939] — test ground truth labels

# Print samples and shapes for verification
print(train_data[:5])
print(train_labels[:5])
print(train_data.shape)    # torch.Size([3830, 3])
print(test_data[:5])
print(test_labels[:5])
print(test_data.shape)     # torch.Size([8939, 3])

tensor([[ 21,  31, 113],
        [ 30,  98, 113],
        [ 47,  10, 113],
        [ 86,  21, 113],
        [ 99,  83, 113]], device='cuda:0')
tensor([ 52,  15,  57, 107,  69], device='cuda:0')
torch.Size([3830, 3])
tensor([[ 43,  40, 113],
        [ 31,  42, 113],
        [ 39,  63, 113],
        [ 35,  61, 113],
        [112, 102, 113]], device='cuda:0')
tensor([ 83,  73, 102,  96, 101], device='cuda:0')
torch.Size([8939, 3])


## Define Model

We use a **tiny 1-layer transformer** built with [TransformerLens](https://github.com/TransformerLensOrg/TransformerLens) — a library that instruments every intermediate activation with **hook points**, enabling us to extract and analyze any internal computation.

Architecture:
```
Input tokens [a, b, =]
  → Embedding (W_E): token → 128-dim vector         [114 × 128]
  → + Positional Embedding: adds position info       [3 × 128]
  → Attention Layer: 4 heads, each 32-dim            [128 → 32 → 128 per head]
  → + Residual connection
  → MLP: Linear(128→512) → ReLU → Linear(512→128)
  → + Residual connection
  → Unembedding (W_U): 128-dim → 113 logits          [128 × 113]
```

Key design choices for interpretability:
- **No LayerNorm** (`normalization_type=None`) — removes a nonlinearity that complicates analysis
- **`d_vocab=p+1=114`** — 113 numbers + 1 equals sign token
- **`d_vocab_out=p=113`** — only predicts numbers (not the = token)
- **`n_ctx=3`** — only 3 positions needed

In [14]:
# --- Model Architecture Configuration ---
# A tiny 1-layer transformer designed for interpretability.
cfg = HookedTransformerConfig(
    n_layers = 1,              # Just 1 transformer block (attention + MLP)
    n_heads = 4,               # 4 attention heads
    d_model = 128,             # Residual stream width (embedding dimension)
    d_head = 32,               # Per-head dimension: d_model / n_heads = 128 / 4 = 32
    d_mlp = 512,               # MLP hidden layer size (4x d_model, standard ratio)
    act_fn = "relu",           # ReLU activation in MLP (not GELU — simpler for interpretation)
    normalization_type=None,   # NO LayerNorm! Removes a nonlinearity that complicates analysis.
    d_vocab=p+1,               # 114 input tokens: {0, 1, ..., 112} for numbers + {113} for '='
    d_vocab_out=p,             # 113 output logits: only predicts numbers (not the '=' token)
    n_ctx=3,                   # Context window = 3 positions: [a, b, =]
    init_weights=True,         # Use standard random initialization for weights
    device=device,             # Place model on GPU (if available) or CPU
    seed = 999,                # Random seed for reproducible weight initialization
)

In [15]:
# --- Instantiate the Model ---
# HookedTransformer automatically inserts hook points at every intermediate computation
# (embedding, attention QKV, attention pattern, MLP pre/post activations, residual stream, etc.)
# This enables run_with_cache() to record ALL internal activations during a forward pass.
model = HookedTransformer(cfg)

Freeze all bias parameters at zero. Biases (`b_Q`, `b_K`, `b_V`, `b_O`, `b_in`, `b_out`, etc.) aren't needed for this symmetric task and would add free parameters that complicate interpretation. Setting `requires_grad = False` prevents them from being updated during training.

In [16]:
# --- Freeze All Bias Parameters at Zero ---
# Iterate over every named parameter in the model.
# Any parameter with "b_" in its name is a bias term (b_Q, b_K, b_V, b_O, b_in, b_out, etc.)
# Setting requires_grad=False prevents the optimizer from updating these during training.
# Result: biases stay at their initialized value (zero), reducing degrees of freedom
# and making the model easier to interpret (only weight matrices matter).
for name, param in model.named_parameters():
    if "b_" in name:
        param.requires_grad = False

**Loss function**: Standard cross-entropy, but with two important details:
1. `logits[:, -1]` — we take logits at the **last sequence position** (the `=` token). The model predicts the answer at this position.
2. `logits.to(torch.float64)` — uses double precision for numerical stability in `log_softmax`.

The initial loss should be ~`ln(113) ≈ 4.727` — the cross-entropy of a **uniform distribution** over 113 classes. An untrained model is essentially guessing randomly.

## Define Optimizer + Loss

**AdamW** with weight decay **1.0** — this is extremely strong regularization. The weight decay term adds `wd * ||weights||²` to the loss, continuously shrinking all weights. This is what drives grokking:
- The **memorization solution** requires large weights spread across many Fourier modes → heavily penalized
- The **generalizing Fourier circuit** is compact (uses only 5 frequencies) → less penalized
- Over time, weight decay erodes memorization faster than it erodes the circuit

In [17]:
# --- Create AdamW Optimizer ---
# AdamW = Adam with DECOUPLED weight decay (applies weight decay directly to weights,
# not through the gradient). This is important for grokking:
# - Each step, weights are multiplied by (1 - lr * wd) = (1 - 0.001 * 1.0) = 0.999
# - This continuously shrinks ALL weights, penalizing large memorization weights more than
#   the compact Fourier circuit weights
optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd, betas=betas)

In [18]:
# --- Loss Function: Cross-Entropy Loss ---
def loss_fn(logits, labels):
    if len(logits.shape)==3:
        logits = logits[:, -1]  # If shape is [batch, seq, vocab], take only the LAST position (the '=' token)
                                # This is where the model makes its prediction. → [batch, vocab]
    logits = logits.to(torch.float64)  # Cast to float64 for numerical stability in log_softmax
    log_probs = logits.log_softmax(dim=-1)  # Compute log(softmax(logits)) along vocab dimension
                                             # More numerically stable than log(softmax(...)) separately
    correct_log_probs = log_probs.gather(dim=-1, index=labels[:, None])[:, 0]
    # labels[:, None] reshapes [batch] → [batch, 1] for gather
    # .gather(dim=-1, ...) picks the log-probability of the CORRECT class for each example
    # [:, 0] removes the extra dim → [batch]
    return -correct_log_probs.mean()  # Negative mean log-prob = cross-entropy loss (scalar)

# --- Verify Initial Loss ---
# An untrained model outputs roughly uniform logits → loss ≈ ln(113) ≈ 4.727
train_logits = model(train_data)              # Forward pass on training data: [3830, 3, 113]
train_loss = loss_fn(train_logits, train_labels)
print(train_loss)                              # Should be ~4.73 (random guessing)
test_logits = model(test_data)                # Forward pass on test data
test_loss = loss_fn(test_logits, test_labels)
print(test_loss)                               # Should also be ~4.73

tensor(4.7359, device='cuda:0', dtype=torch.float64, grad_fn=<NegBackward0>)
tensor(4.7330, device='cuda:0', dtype=torch.float64, grad_fn=<NegBackward0>)


In [19]:
# --- Verify: Expected Loss for Uniform Random Guessing ---
# If the model assigns equal probability 1/113 to all classes, loss = -log(1/113) = log(113)
print("Uniform loss:")
print(np.log(p))  # ln(113) ≈ 4.7274 — should match the initial loss above

Uniform loss:
4.727387818712341


## Actually Train

**Design choice:** Full-batch training (all 3,830 training examples at once) instead of stochastic gradient descent. This makes training smoother and reduces "slingshots". We save a deep copy of model weights every 100 epochs to analyze how the model evolves.

In [ ]:
# --- Training Loop ---
train_losses = []          # Will store train loss at every epoch (25,000 values)
test_losses = []           # Will store test loss at every epoch
model_checkpoints = []     # Will store deep copies of model weights every 100 epochs
checkpoint_epochs = []     # Will store which epoch each checkpoint came from

if TRAIN_MODEL:
    for epoch in tqdm.tqdm(range(num_epochs)):  # 25,000 epochs with progress bar

        # --- Forward pass on ALL training data (full-batch, not mini-batch) ---
        train_logits = model(train_data)              # [3830, 3, 113] — model output
        train_loss = loss_fn(train_logits, train_labels)  # Scalar cross-entropy loss
        train_loss.backward()                          # Backprop: compute gradients for all parameters
        train_losses.append(train_loss.item())         # Record loss value (detached from computation graph)

        optimizer.step()       # Apply gradients + weight decay to update model weights
        optimizer.zero_grad()  # Reset accumulated gradients for next iteration

        # --- Evaluate on test set (no gradient computation needed) ---
        with torch.inference_mode():  # Disables autograd for efficiency
            test_logits = model(test_data)              # [8939, 3, 113]
            test_loss = loss_fn(test_logits, test_labels)
            test_losses.append(test_loss.item())

        # --- Save checkpoint every 100 epochs ---
        if ((epoch+1)%checkpoint_every)==0:  # At epoch 99, 199, 299, ... (250 checkpoints total)
            checkpoint_epochs.append(epoch)
            model_checkpoints.append(copy.deepcopy(model.state_dict()))
            # copy.deepcopy is CRITICAL: state_dict() returns references to live tensors
            # that change every step. deepcopy creates an independent snapshot.
            print(f"Epoch {epoch} Train Loss {train_loss.item()} Test Loss {test_loss.item()}")

  0%|          | 0/25000 [00:00<?, ?it/s]

Epoch 99 Train Loss 2.9216481191413375 Test Loss 7.6559031896872165
Epoch 199 Train Loss 0.030839887431076563 Test Loss 19.93917900257167
Epoch 299 Train Loss 0.009591797266535414 Test Loss 20.63647831249946
Epoch 399 Train Loss 0.0030786268437847614 Test Loss 21.75797263942504
Epoch 499 Train Loss 0.001020158643209917 Test Loss 22.978755738221764
Epoch 599 Train Loss 0.00034355147184966903 Test Loss 24.253354576350326
Epoch 699 Train Loss 0.0001173767403831185 Test Loss 25.524252550329265
Epoch 799 Train Loss 4.073800109283013e-05 Test Loss 26.799500937218017
Epoch 899 Train Loss 1.4425790894557507e-05 Test Loss 28.049518731389735
Epoch 999 Train Loss 5.31517889310081e-06 Test Loss 29.23085039873119
Epoch 1099 Train Loss 2.1192837692262375e-06 Test Loss 30.277232977110355


In [ ]:
# --- Save Everything to Disk ---
# Saves the final model weights, all 250 intermediate checkpoints, loss curves,
# and the train/test split indices for full reproducibility.
# Only saves when we just trained — otherwise we'd overwrite good data with empty lists.
if TRAIN_MODEL:
    torch.save(
        {
            "model": model.state_dict(),              # Final trained model weights (dict of tensors)
            "config": model.cfg,                       # Model architecture config (HookedTransformerConfig)
            "checkpoints": model_checkpoints,          # List of 250 state_dicts (one every 100 epochs)
            "checkpoint_epochs": checkpoint_epochs,    # List of epoch numbers: [99, 199, ..., 24999]
            "test_losses": test_losses,                # 25,000 test loss values (one per epoch)
            "train_losses": train_losses,              # 25,000 train loss values
            "train_indices": train_indices,            # Which of the 12,769 examples are in the training set
            "test_indices": test_indices,              # Which are in the test set
        },
        PTH_LOCATION)  # Saves to "workspace/_scratch/grokking_demo.pth"

> **📋 Diff from original:** The original uses `torch.load(PTH_LOCATION)` without the `weights_only` parameter. This version adds `weights_only=False` for PyTorch ≥2.6 compatibility, which changed the default from `False` to `True`.


In [ ]:
# --- Load from Checkpoint (if not training from scratch) ---
if not TRAIN_MODEL:
    cached_data = torch.load(PTH_LOCATION, weights_only=False)
    # weights_only=False: allows loading arbitrary Python objects (needed for the config dataclass)
    model.load_state_dict(cached_data['model'])           # Restore final model weights
    model_checkpoints = cached_data["checkpoints"]        # Restore 250 intermediate checkpoints
    checkpoint_epochs = cached_data["checkpoint_epochs"]  # Restore checkpoint epoch numbers
    test_losses = cached_data['test_losses']              # Restore test loss history
    train_losses = cached_data['train_losses']            # Restore train loss history
    train_indices = cached_data["train_indices"]          # Restore train/test split
    test_indices = cached_data["test_indices"]

## Training Curves — Observe Grokking!

The plot below shows the classic grokking pattern:
- **Train loss** (blue) drops to near-zero within ~1,500 epochs → the model has **memorized** the training set
- **Test loss** (red) stays at ~ln(113) ≈ 4.73 (random guessing) for thousands more epochs
- **Test loss suddenly drops** around epoch ~13,000 → the model has finally **generalized** (grokking!)

The big question: what happens internally between epochs 1,500 and 13,000? The rest of this notebook answers that.

In [ ]:
# --- Install Neel Nanda's Custom Plotly Library ---
# Provides enhanced line() function with features like line_labels, toggle_x, toggle_y, return_fig
%pip install git+https://github.com/neelnanda-io/neel-plotly.git

# Re-import line from neel_plotly (overrides the simple wrapper defined in cell 12)
from neel_plotly.plot import line

# --- Plot Training Curves (Log Scale) ---
# Subsample every 100th loss value (250 points from 25,000) for cleaner visualization
line(
    [train_losses[::100], test_losses[::100]],   # Two lines: train (blue) and test (red)
    x=np.arange(0, len(train_losses), 100),      # X-axis: epoch numbers [0, 100, 200, ..., 24900]
    xaxis="Epoch", yaxis="Loss",
    log_y=True,                                   # Log scale on Y-axis — essential for seeing both phases
    title="Training Curve for Modular Addition",
    line_labels=['train', 'test'],                # Legend labels
    toggle_x=True, toggle_y=True                  # Interactive buttons to toggle log/linear scale
)
# Expected result: train loss drops by ~epoch 1500, test loss drops much later (~epoch 13000) = GROKKING

# Analysing the Model

Now that the model has grokked, we reverse-engineer **what algorithm** it learned. The key tool is `model.run_with_cache()` — this runs the forward pass while recording every intermediate activation (embeddings, attention patterns, MLP activations, residual stream states) at hook points throughout the model.

## Standard Things to Try

First, run the entire dataset (all 12,769 input pairs) through the model and cache all activations. `run_with_cache` returns:
- `original_logits`: shape `[12769, 3, 113]` — the model's output logits at each position
- `cache`: an `ActivationCache` object — a dict-like container where keys are hook names (e.g. `"blocks.0.attn.hook_pattern"`) and values are the corresponding activation tensors

In [ ]:
# --- Run the Full Dataset Through the Model, Caching All Activations ---
# run_with_cache() performs a forward pass and records every intermediate activation at hook points.
# Returns:
#   original_logits: [12769, 3, 113] — model output logits at every position for every input
#   cache: ActivationCache — dict-like container with keys like "blocks.0.attn.hook_pattern"
original_logits, cache = model.run_with_cache(dataset)
print(original_logits.numel())  # Total elements: 12769 * 3 * 113 = 4,328,721

Extract the key **composed weight matrices** that define the model's computation paths:

- **`W_E`** `[113, 128]`: The embedding matrix (excluding row 113, the `=` token — we only care about number embeddings). Maps each number 0–112 to a 128-dim vector.
- **`W_neur`** `[4, 113, 512]`: The full path **input token → neuron pre-activation** through one attention head: `W_E @ W_V @ W_O @ W_in`. For each of the 4 heads, tells us how each input token affects each of the 512 MLP neurons.
- **`W_logit`** `[512, 113]`: The path **neuron activation → output logit**: `W_out @ W_U`. Each row tells us how one MLP neuron contributes to all 113 output logits.

In [ ]:
# --- Extract Composed Weight Matrices ---
# These matrices represent end-to-end computation paths through the model.

# W_E: Embedding matrix for numbers only (exclude the '=' token at index 113)
# Shape: [113, 128] — maps each number (0-112) to a 128-dim embedding vector
W_E = model.embed.W_E[:-1]
print("W_E", W_E.shape)  # [113, 128]

# W_neur: Full path from input token to MLP neuron pre-activation, through one attention head.
# Composition: W_E @ W_V @ W_O @ W_in
#   W_E [113, 128]  — token → embedding
#   W_V [4, 128, 32] — embedding → value vector (per head)
#   W_O [4, 32, 128]  — attention output → residual stream (per head)
#   W_in [128, 512]   — residual stream → MLP hidden layer
# Result: [4, 113, 512] — for each head, how each input token affects each MLP neuron
W_neur = W_E @ model.blocks[0].attn.W_V @ model.blocks[0].attn.W_O @ model.blocks[0].mlp.W_in
print("W_neur", W_neur.shape)  # [4, 113, 512]

# W_logit: Path from MLP neuron activation to output logit.
# Composition: W_out @ W_U
#   W_out [512, 128] — MLP hidden → residual stream
#   W_U [128, 113]   — residual stream → output logits (unembedding)
# Result: [512, 113] — how each MLP neuron contributes to each of the 113 output logits
W_logit = model.blocks[0].mlp.W_out @ model.unembed.W_U
print("W_logit", W_logit.shape)  # [512, 113]

In [ ]:
# --- Compute Loss on the Full Dataset ---
# After grokking, loss should be very small (~1e-6) — the model predicts nearly perfectly.
original_loss = loss_fn(original_logits, labels).item()
print("Original Loss:", original_loss)

### Looking at Activations

We extract specific activations from the cache. The cache uses a tuple-key system: `cache["activation_type", layer_index, "sublayer"]`.

Extract key activations at the **last position** (the `=` token, position index 2) — this is where the model makes its prediction:

- **`pattern_a`** `[12769, 4]`: Attention weight from `=` → `a` for each head. `cache["pattern", 0, "attn"]` has shape `[batch, heads, query_pos, key_pos]`, so `[:, :, -1, 0]` selects query=last(=), key=first(a).
- **`pattern_b`** `[12769, 4]`: Attention weight from `=` → `b` (key=second position).
- **`neuron_acts`** `[12769, 512]`: Post-ReLU MLP activations at position `=`. These are the neurons that directly compute the answer.
- **`neuron_pre_acts`** `[12769, 512]`: Pre-ReLU activations (before the nonlinearity).

### Attention Pattern Visualizations

The plots below show how each attention head distributes attention when the `=` token (position 2) attends to `a` (position 0), `b` (position 1), and itself. The paper finds that heads specialize: some attend primarily to `a`, others to `b`, and some attend roughly equally to both. This is how the model moves information from `a` and `b` to the `=` position where the prediction is made.

In [ ]:
# --- Extract Key Activations at the '=' Position ---
# The model makes its prediction at position 2 (the '=' token), so we extract activations there.

# Attention weight from '=' (position 2) → 'a' (position 0), for each head
# cache["pattern", 0, "attn"] shape: [12769, 4, 3, 3] = [batch, heads, query_pos, key_pos]
# [:, :, -1, 0] = query=last(=), key=first(a) → [12769, 4]
pattern_a = cache["pattern", 0, "attn"][:, :, -1, 0]

# Attention weight from '=' → 'b' (position 1), for each head → [12769, 4]
pattern_b = cache["pattern", 0, "attn"][:, :, -1, 1]

# Post-ReLU MLP activations at the '=' position
# cache["post", 0, "mlp"] shape: [12769, 3, 512] = [batch, position, neuron]
# [:, -1, :] selects position=2 (=) → [12769, 512]
neuron_acts = cache["post", 0, "mlp"][:, -1, :]

# Pre-ReLU MLP activations at the '=' position → [12769, 512]
neuron_pre_acts = cache["pre", 0, "mlp"][:, -1, :]

Print all cached activation shapes — this reveals the full computational graph of the 1-layer transformer:

| Hook name | Shape | Description |
|-----------|-------|-------------|
| `hook_embed` | `[12769, 3, 128]` | Token embeddings (lookup from W_E) |
| `hook_pos_embed` | `[12769, 3, 128]` | Positional embeddings |
| `blocks.0.hook_resid_pre` | `[12769, 3, 128]` | Residual stream input (embed + pos_embed) |
| `blocks.0.attn.hook_q/k/v` | `[12769, 3, 4, 32]` | Query/Key/Value vectors per head |
| `blocks.0.attn.hook_attn_scores` | `[12769, 4, 3, 3]` | Raw attention scores (pre-softmax) |
| `blocks.0.attn.hook_pattern` | `[12769, 4, 3, 3]` | Attention pattern (post-softmax) |
| `blocks.0.attn.hook_z` | `[12769, 3, 4, 32]` | Attention output per head |
| `blocks.0.hook_attn_out` | `[12769, 3, 128]` | Combined multi-head attention output |
| `blocks.0.hook_resid_mid` | `[12769, 3, 128]` | Residual stream after attention, before MLP |
| `blocks.0.mlp.hook_pre/post` | `[12769, 3, 512]` | MLP pre/post-ReLU activations |
| `blocks.0.hook_mlp_out` | `[12769, 3, 128]` | MLP output projected back to residual stream |
| `blocks.0.hook_resid_post` | `[12769, 3, 128]` | Final residual stream (→ unembedding → logits) |

In [ ]:
# --- Print All Cached Activation Shapes ---
# This reveals every hook point in the 1-layer transformer's computation graph.
# Each entry is a (name, tensor) pair showing what was recorded during the forward pass.
for param_name, param in cache.items():
    print(param_name, param.shape)

In [ ]:
# --- Average Attention Pattern per Head ---
# cache["pattern", 0, "attn"] shape: [12769, 4, 3, 3] = [batch, heads, query_pos, key_pos]
# cache["pattern", 0]: [12769, 4, 3, 3] — attention patterns for layer 0
# .mean(dim=0): average over all 12,769 inputs → [4, 3, 3] — one pattern per head
# [:, -1, :]: query = '=' (position 2), all keys → [4, 3]
# Shows: for each head, how much '=' attends to 'a', 'b', and itself (averaged over all inputs)
imshow(cache["pattern", 0].mean(dim=0)[:, -1, :], title="Average Attention Pattern per Head", xaxis="Source", yaxis="Head", x=['a', 'b', '='])

In [ ]:
# --- Attention Pattern for a Single Input Example ---
# cache["pattern", 0, "attn"] shape: [12769, 4, 3, 3] = [batch, heads, query_pos, key_pos]
# [5] picks the 6th input: (a=0, b=5) — shows attention for this specific input pair.
# [:, -1, :] selects query='=' for all 4 heads → [4, 3]
imshow(cache["pattern", 0][5][:, -1, :], title="Attention Pattern for a Single Input Example", xaxis="Source", yaxis="Head", x=['a', 'b', '='])

In [ ]:
# --- Inspect First 4 Input Sequences ---
# Verify: [0,0,113], [0,1,113], [0,2,113], [0,3,113]
dataset[:4]

In [ ]:
# --- Head 0 Attention to 'a' as a Heatmap over All (a,b) Pairs ---
# [:, 0, -1, 0]: head=0, query='=', key='a' → [12769] (one value per input pair)
# .reshape(p, p): reshape to [113, 113] grid indexed by (a, b)
# Shows how head 0's attention to 'a' varies across all input combinations
imshow(cache["pattern", 0][:, 0, -1, 0].reshape(p, p), title="Attention for Head 0 from a -> =", xaxis="b", yaxis="a")

In [ ]:
# --- All 4 Heads' Attention to 'a', Side by Side ---
# [:, :, -1, 0]: all heads, query='=', key='a' → [12769, 4]
# rearrange "(a b) head -> head a b": reshape batch to (a, b) grid → [4, 113, 113]
# facet_col=0: display as 4 side-by-side subplots (one heatmap per head)
imshow(
    einops.rearrange(cache["pattern", 0][:, :, -1, 0], "(a b) head -> head a b", a=p, b=p),
    title="Attention for Head 0 from a -> =", xaxis="b", yaxis="a", facet_col=0)

### Neuron Activations

Reshape the flat batch dimension `[12769]` back to a 2D grid `[113, 113]` indexed by `(a, b)` using `einops.rearrange`. This lets us visualize each neuron's activation as a **heatmap over all (a, b) input pairs**. If the neurons have learned something structured, we should see patterns rather than noise.

In [ ]:
# --- Check MLP Activation Shape ---
# [12769, 3, 512] = [batch, position, neuron]
# 12,769 inputs × 3 positions × 512 MLP neurons
cache["post", 0, "mlp"].shape

In [ ]:
# --- Visualize First 5 Neurons as 2D Heatmaps ---
# neuron_acts[:, :5]: first 5 neurons' activations at '=' position → [12769, 5]
# rearrange "(a b) neuron -> neuron a b": reshape flat batch to (a, b) grid → [5, 113, 113]
# facet_col=0: display 5 side-by-side heatmaps (one per neuron)
# If the model learned a Fourier algorithm, these should show periodic diagonal stripe patterns
imshow(
    einops.rearrange(neuron_acts[:, :5], "(a b) neuron -> neuron a b", a=p, b=p),
    title="First 5 neuron acts", xaxis="b", yaxis="a", facet_col=0)

### Singular Value Decomposition

SVD of the embedding matrix `W_E` reveals its **effective dimensionality**. A random 113×128 matrix would have roughly uniform singular values. But if the learned embedding is structured (using only a few Fourier modes), it will have a few large singular values and many near-zero ones. The principal components (columns of U) show the dominant directions in input space — and they look like **sine and cosine waves**, our first clue that the model uses Fourier representations.

In [ ]:
# --- Check Embedding Matrix Shape ---
# [113, 128] — 113 number tokens, each mapped to a 128-dimensional embedding vector
W_E.shape

In [ ]:
# --- Singular Value Decomposition of the Embedding ---
# SVD decomposes W_E [113, 128] into U [113, 113] × diag(S [113]) × Vh^T [128, 113]
# U columns = principal components in token space (directions that capture most variance)
# S = singular values (importance of each component, in decreasing order)
# Vh rows = principal components in embedding space
U, S, Vh = torch.svd(W_E)

# Plot singular values: if only a few are large, the embedding is low-rank (uses few dimensions)
line(S, title="Singular Values")

# Plot U: each column is a direction in the 113-token space.
# For a Fourier embedding, these columns should look like sine/cosine waves.
imshow(U, title="Principal Components on the Input")

The principal components (top columns of U from SVD) plotted against input token index look like **sine and cosine waves** at specific frequencies. This is the first major clue that the embedding encodes numbers as Fourier components.

Now let's formalize this by constructing an explicit **Fourier basis** for ℤ₁₁₃ and measuring how the embedding projects onto it. The Fourier basis for ℝᵖ consists of:
- 1 constant vector
- For each frequency k = 1, ..., p//2: one `sin(2πkn/p)` vector and one `cos(2πkn/p)` vector
- Total: 1 + 2×56 = 113 orthonormal basis vectors

In [ ]:
# --- Control: SVD of a Random Matrix ---
# Compare with a random Gaussian matrix of the same shape.
# Random matrices have roughly UNIFORM singular values and noisy principal components.
# This contrast shows that the learned embedding is genuinely structured (not random).
U, S, Vh = torch.svd(torch.randn_like(W_E))
line(S, title="Singular Values Random")
imshow(U, title="Principal Components Random")

> **📋 Diff from original:** The original section header is `## Explaining Algorithm`. Renamed here to `## Reverse-Engineering the Algorithm` for clarity.


## Reverse-Engineering the Algorithm

The SVD revealed that the embedding's principal components are sinusoidal. This section formalizes and confirms that the model implements a **Discrete Fourier Transform (DFT) based algorithm** for modular addition.

**Core idea**: The integers mod p can be mapped onto a circle. Addition mod p corresponds to **rotation** on that circle. Trig functions (sin and cos) are the natural basis for representing rotations, and the trig angle-addition identities let you compute `f(a+b)` from `f(a)` and `f(b)`.

> **📋 Diff from original:** The original header is `### Analyse the Embedding — It's a Lookup Table!`. Renamed to `### The Embedding is a Fourier Lookup Table`.


### The Embedding is a Fourier Lookup Table

The top principal components of `W_E` look sinusoidal. This suggests the embedding maps each token `a` to a vector whose components are `sin(wₖa)` and `cos(wₖa)` at various frequencies `wₖ = 2πk/113`. Let's verify this by projecting onto an explicit Fourier basis.

In [ ]:
# --- Plot Top 8 Principal Components of the Embedding ---
# U[:, :8].T: transpose first 8 columns of U → [8, 113]
# Each row is a principal component plotted against token index (0–112).
# These should look like sine and cosine waves at specific frequencies — our first clue!
U, S, Vh = torch.svd(W_E)
line(U[:, :8].T, title="Principal Components of the embedding", xaxis="Input Vocabulary")

In [ ]:
# --- Construct the Discrete Fourier Basis for Z_113 ---
# The Fourier basis for R^p consists of:
#   - 1 constant vector (frequency 0)
#   - For each frequency k = 1, ..., p//2: one sin(2πkn/p) and one cos(2πkn/p) vector
#   - Total: 1 + 2 × 56 = 113 orthonormal basis vectors (complete basis for R^113)

fourier_basis = []
fourier_basis_names = []

# Frequency 0: constant vector (all ones)
fourier_basis.append(torch.ones(p))
fourier_basis_names.append("Constant")

for freq in range(1, p//2+1):  # freq = 1, 2, ..., 56
    # Sin component: sin(2π·freq·n/113) for n = 0, 1, ..., 112
    fourier_basis.append(torch.sin(torch.arange(p)*2 * torch.pi * freq / p))
    fourier_basis_names.append(f"Sin {freq}")
    # Cos component: cos(2π·freq·n/113) for n = 0, 1, ..., 112
    fourier_basis.append(torch.cos(torch.arange(p)*2 * torch.pi * freq / p))
    fourier_basis_names.append(f"Cos {freq}")

# Stack into a [113, 113] matrix and normalize each row to unit length
fourier_basis = torch.stack(fourier_basis, dim=0).to(device)          # [113, 113]
fourier_basis = fourier_basis/fourier_basis.norm(dim=-1, keepdim=True) # Normalize → orthonormal basis

# Visualize: each row is a basis vector (sine or cosine wave at a different frequency)
imshow(fourier_basis, xaxis="Input", yaxis="Component", y=fourier_basis_names)

In [ ]:
# --- Visualize Individual Fourier Basis Vectors ---
# First 8: constant, sin(1), cos(1), sin(2), cos(2), sin(3), cos(3), sin(4) — low frequency, smooth
line(fourier_basis[:8], xaxis="Input", line_labels=fourier_basis_names[:8], title="First 8 Fourier Components")
# Middle 4: sin(13), cos(13), sin(14), cos(14) — higher frequency, more oscillations
line(fourier_basis[25:29], xaxis="Input", line_labels=fourier_basis_names[25:29], title="Middle Fourier Components")

In [ ]:
# --- Verify Orthonormality of Fourier Basis ---
# fourier_basis @ fourier_basis.T: [113, 113] inner product (Gram) matrix.
# If the basis is orthonormal, this should be the identity matrix:
#   diagonal = 1 (each vector has unit norm)
#   off-diagonal ≈ 0 (all vectors are orthogonal to each other)
imshow(fourier_basis @ fourier_basis.T, title="All Fourier Vectors are Orthogonal")

### Embedding in the Fourier Basis

Now we project the embedding matrix `W_E` into our Fourier basis by computing `fourier_basis @ W_E`. Each row of the result shows how much one Fourier component (sin or cos at some frequency) contributes to the embedding's 128-dimensional representation.

If the model uses only a few key frequencies, this matrix will be **sparse** — most rows will be near-zero, with a few rows (the key frequencies) having large norms.

In [ ]:
# --- Project Embedding into Fourier Basis ---
# fourier_basis @ W_E: [113, 113] @ [113, 128] → [113, 128]
# Row k shows how much Fourier component k contributes to each of the 128 embedding dimensions.
# If the model uses only a few key frequencies, most rows will be near-zero.
imshow(fourier_basis @ W_E, yaxis="Fourier Component", xaxis="Residual Stream", y=fourier_basis_names, title="Embedding in Fourier Basis")

In [ ]:
# --- Fourier Norms of the Embedding ---
# .norm(dim=-1): L2 norm of each row of (fourier_basis @ W_E) → [113]
# One value per Fourier component — higher = more energy allocated to that frequency.
# This reveals WHICH frequencies the embedding uses: expect sharp peaks at the 5 key frequencies.
line((fourier_basis @ W_E).norm(dim=-1), xaxis="Fourier Component", x=fourier_basis_names, title="Norms of Embedding in Fourier Basis")

> **📋 Diff from original:** Two changes here:
> 1. **Different key frequencies:** Original has `key_freqs = [17, 25, 32, 47]` (4 freqs). This version has `key_freqs = [9, 33, 36, 38, 55]` (5 freqs) — due to a different training run/random seed.
> 2. **Dynamic index computation:** Original hardcodes `key_freq_indices = [33, 34, 49, 50, 63, 64, 93, 94]`. This version computes them dynamically: `key_freq_indices = [i for f in key_freqs for i in (2*f - 1, 2*f)]`.


In [ ]:
# --- Identify Key Frequencies ---
# The norms plot above shows 5 peaks at these specific frequencies:
key_freqs = [9, 33, 36, 38, 55]

# For each frequency f, the Fourier basis has sin at index 2f-1 and cos at index 2f.
# Collect all 10 indices (sin + cos for each of the 5 key frequencies):
key_freq_indices = [i for f in key_freqs for i in (2*f - 1, 2*f)]
# key_freq_indices = [17, 18, 65, 66, 71, 72, 75, 76, 109, 110]

# Extract the 10 key Fourier components of the embedding
fourier_embed = fourier_basis @ W_E          # [113, 128] — full projection
key_fourier_embed = fourier_embed[key_freq_indices]  # [10, 128] — only key components
print("key_fourier_embed", key_fourier_embed.shape)

# --- Verify Orthogonality of Key Components ---
# [10, 128] @ [128, 10] → [10, 10] Gram matrix
# Should be nearly diagonal: each frequency's sin/cos occupy their own orthogonal subspace
imshow(key_fourier_embed @ key_fourier_embed.T, title="Dot Product of embedding of key Fourier Terms")

### Key Frequencies

The norms plot above shows sharp peaks at 5 specific frequencies: **k ∈ {9, 33, 36, 38, 55}**. These are the **key frequencies** — the model has allocated dedicated dimensions in its 128-dim embedding space to represent `sin(wₖa)` and `cos(wₖa)` for each of these frequencies (10 Fourier components total, 2 per frequency).

The dot product matrix below confirms these 10 components are nearly **orthogonal** in embedding space — the model has cleanly separated each frequency into its own subspace.

> **📋 Diff from original:** The original hardcodes cosine indices as `fourier_basis[[34, 50, 64, 94]]`. This version computes them dynamically: `fourier_basis[[2*f for f in key_freqs]]`.


In [ ]:
# --- Visualize the 5 Key Cosine Waves ---
# cos_indices: indices into fourier_basis for cos components of key frequencies
cos_indices = [2*f for f in key_freqs]  # [18, 66, 72, 76, 110]
# Plot these 5 cosine waves — each has a different period
line(fourier_basis[cos_indices], title="Cos of key freqs", line_labels=cos_indices)

> **📋 Diff from original:** Same as above — the original uses `fourier_basis[[34, 50, 64, 94]].mean(0)` with hardcoded indices. This version uses `fourier_basis[[2*f for f in key_freqs]].mean(0)`.


In [ ]:
# --- Demonstrate Constructive Interference ---
# Average the 5 cosine waves at the key frequencies.
# At position 0: all cos waves = 1 → average = 1 (CONSTRUCTIVE interference)
# At other positions: waves oscillate at different frequencies → cancel out (DESTRUCTIVE interference)
# Result: a sharp spike at position 0 — this is how the model creates a peak at the correct answer!
# In the model: sum_k cos(w_k(a+b-c)) peaks when c = (a+b) mod p.
line(fourier_basis[[2*f for f in key_freqs]].mean(0), title="Constructive Interference")

**Why constructive interference works**: Each `cos(wₖ·c)` wave equals 1 at `c = 0` and oscillates elsewhere. When we average cosines at 5 different frequencies, they all agree at `c = 0` (all equal 1 → constructive interference) but cancel each other out at other values of c (destructive interference). This creates a sharp peak at the correct answer.

In the model's output: `Σₖ cos(wₖ(a+b−c))` peaks sharply when `c ≡ a+b (mod p)` — the sum of 5 cosine waves creates a "spike" at the right answer.

## Analyse Neurons

Now we investigate what the 512 MLP neurons compute. According to the paper's theory, each neuron should compute `cos(wₖ(a+b))` or `sin(wₖ(a+b))` for one of the key frequencies, using the trig identity:

$$\cos(w_k(a+b)) = \cos(w_k a)\cos(w_k b) - \sin(w_k a)\sin(w_k b)$$

The attention layer copies the Fourier-encoded embeddings of `a` and `b` to the `=` position, giving the MLP access to `sin(wₖa)`, `cos(wₖa)`, `sin(wₖb)`, `cos(wₖb)`. The ReLU neurons then compute the products and differences needed for the trig identity.

In [ ]:
# --- Neuron Activations as 2D Heatmaps (Repeated for Context) ---
# Same visualization as cell 60: first 5 neurons reshaped to (a, b) grid → [5, 113, 113]
imshow(
    einops.rearrange(neuron_acts[:, :5], "(a b) neuron -> neuron a b", a=p, b=p),
    title="First 5 neuron acts", xaxis="b", yaxis="a", facet_col=0)

In [ ]:
# --- Single Neuron Heatmap (Neuron 0) ---
# Reshape neuron 0's activation from flat [12769] to grid [113, 113] indexed by (a, b).
# If neuron 0 computes cos(w_k(a+b)), the heatmap should show diagonal stripes
# along lines of constant (a+b), with period determined by frequency k.
imshow(
    einops.rearrange(neuron_acts[:, 0], "(a b) -> a b", a=p, b=p),
    title="First neuron act", xaxis="b", yaxis="a",)

**Visualizing the trig identity**: The plots below show `cos(wₖa) × cos(wₖb)` — one of the product terms in the trig identity `cos(wₖ(a+b)) = cos(wₖa)cos(wₖb) − sin(wₖa)sin(wₖb)`. The MLP neurons effectively compute these products and combine them to produce `cos(wₖ(a+b))`.

The 2D Fourier transform of each neuron's activation confirms this: the energy is concentrated at the specific frequency the neuron is tuned to.

> **📋 Diff from original:** The original hardcodes frequency 47 (index 94) for the trig identity visualization. This version uses `key_freqs[0]` (= 9, index 18) so it adapts to whichever frequencies the model learned.


In [ ]:
# --- Visualize cos(w_k·a) × cos(w_k·b) — One Term of the Trig Identity ---
# The MLP computes cos(w_k(a+b)) = cos(w_k·a)cos(w_k·b) - sin(w_k·a)sin(w_k·b)
# Here we visualize the first product term: cos(w_k·a) × cos(w_k·b) as a 2D heatmap.
example_freq = key_freqs[0]   # = 9 (first key frequency)
cos_idx = 2 * example_freq    # = 18 (index of cos(9) in fourier_basis)
# Outer product: [1, 113] * [113, 1] → [113, 113]
imshow(fourier_basis[cos_idx][None, :] * fourier_basis[cos_idx][:, None], title=f"Cos {example_freq}a * cos {example_freq}b")

In [ ]:
# --- Visualize cos(w_k·a) × constant — Depends Only on 'a' ---
# fourier_basis[0] = constant vector (all entries ~1/sqrt(113) after normalization)
# This shows a pattern that varies with 'a' only — one of the simpler terms
imshow(fourier_basis[cos_idx][None, :] * fourier_basis[0][:, None], title=f"Cos {example_freq}a * const")

In [ ]:
# --- 2D Fourier Transform of Neuron 0's Activation ---
# Apply the Fourier basis to both axes: fourier_basis @ neuron_grid @ fourier_basis.T
# neuron_acts[:, 0].reshape(p, p): reshape from flat [12769] to grid [113, 113]
# Result: [113, 113] where entry (i, j) = Fourier coefficient for component i in 'a' and j in 'b'
# For a neuron computing cos(w_k(a+b)), energy concentrates in a 3×3 block at frequency k.
imshow(fourier_basis @ neuron_acts[:, 0].reshape(p, p) @ fourier_basis.T, title="2D Fourier Transformer of neuron 0", xaxis="b", yaxis="a", x=fourier_basis_names, y=fourier_basis_names)

In [ ]:
# --- 2D Fourier Transform of Neuron 5 ---
# Same analysis for neuron 5 — may respond to a different key frequency than neuron 0.
imshow(fourier_basis @ neuron_acts[:, 5].reshape(p, p) @ fourier_basis.T, title="2D Fourier Transformer of neuron 5", xaxis="b", yaxis="a", x=fourier_basis_names, y=fourier_basis_names)

In [ ]:
# --- Control: 2D Fourier Transform of Random Noise ---
# For comparison, apply the same 2D DFT to random Gaussian noise.
# Random noise has energy spread uniformly across all frequencies (no structure).
# This demonstrates that the neuron patterns above are genuinely structured, not artifacts.
imshow(fourier_basis @ torch.randn_like(neuron_acts[:, 0]).reshape(p, p) @ fourier_basis.T, title="2D Fourier Transformer of RANDOM", xaxis="b", yaxis="a", x=fourier_basis_names, y=fourier_basis_names)

### Neuron Frequency Clustering

To quantify which frequency each neuron responds to, we compute a **2D Fourier transform** of every neuron's activation pattern (as a function of a and b), then measure what fraction of each neuron's total variance is explained by each frequency.

For frequency `k`, the relevant 2D Fourier components are a 3×3 block at indices `{0, 2k-1, 2k} × {0, 2k-1, 2k}` — the constant, sin, and cos components for frequency k in both the a and b dimensions. A neuron computing `cos(wₖ(a+b))` will have nearly all its energy in this block.

The result: **most neurons (>85%) are explained by a single frequency**, confirming that neurons cleanly cluster by frequency.

In [ ]:
# --- Compute 2D Fourier Transform of ALL Neuron Activations ---
# Reshape neuron_acts from [12769, 512] to [512, 113, 113] — one 2D grid per neuron
# Apply fourier_basis from both sides: [113, 113] @ [512, 113, 113] @ [113, 113] → [512, 113, 113]
fourier_neuron_acts = fourier_basis @ einops.rearrange(neuron_acts, "(a b) neuron -> neuron a b", a=p, b=p) @ fourier_basis.T

# Zero out the DC (constant) component — we only care about frequency content, not the mean
fourier_neuron_acts[:, 0, 0] = 0.
print("fourier_neuron_acts", fourier_neuron_acts.shape)  # [512, 113, 113]

In [ ]:
# --- Compute Fraction of Variance Explained by Each Frequency for Each Neuron ---
# neuron_freq_norm[freq, neuron] = fraction of neuron's total 2D Fourier energy at frequency (freq+1)
neuron_freq_norm = torch.zeros(p//2, model.cfg.d_mlp).to(device)  # [56, 512]

for freq in range(0, p//2):  # freq = 0, 1, ..., 55 (representing frequencies 1 through 56)
    # For frequency k = freq+1, the relevant 2D Fourier indices are:
    # {0, 2k-1, 2k} in both dimensions = the constant, sin(k), and cos(k) components
    # This 3×3 block captures all energy at frequency k (e.g., cos(k·a)cos(k·b), sin(k·a)cos(k·b), etc.)
    for x in [0, 2*(freq+1) - 1, 2*(freq+1)]:
        for y in [0, 2*(freq+1) - 1, 2*(freq+1)]:
            neuron_freq_norm[freq] += fourier_neuron_acts[:, x, y]**2  # Sum of squared coefficients

# Normalize by total energy per neuron → fraction of variance explained
# fourier_neuron_acts.pow(2).sum(dim=[-1, -2]): total energy per neuron → [512]
neuron_freq_norm = neuron_freq_norm / fourier_neuron_acts.pow(2).sum(dim=[-1, -2])[None, :]

# Heatmap: rows = frequencies (1–56), columns = neurons (0–511)
# Bright spots show which frequency each neuron responds to — expect sharp clusters
imshow(neuron_freq_norm, xaxis="Neuron", yaxis="Freq", y=torch.arange(1, p//2+1), title="Neuron Frac Explained by Freq")

In [ ]:
# --- Sorted Max Fraction Explained ---
# For each neuron, take the MAX fraction explained across all frequencies.
# Then sort in ascending order. Most neurons should have >85% explained by a single frequency.
# .max(dim=0).values: best frequency match per neuron → [512]
# .sort().values: sort ascending for clean visualization
line(neuron_freq_norm.max(dim=0).values.sort().values, xaxis="Neuron", title="Max Neuron Frac Explained over Freqs")

**Result**: When sorted, most neurons have >85% of their variance explained by a single frequency. Of 512 neurons, the paper reports ~433 (84.6%) are well-approximated by degree-2 polynomials of a single key frequency. The neurons form clean **frequency clusters** — each cluster implements one piece of the Fourier algorithm.

## Neuron → Logit Analysis (The Unembedding)

The final step of the algorithm: how do neuron activations get converted to output logits? The composite matrix `W_logit = W_out @ W_U` maps each neuron's activation directly to the 113 output logits. By projecting `W_logit` into the Fourier basis, we can see which Fourier components appear in the output.

The paper predicts that the output logits should be proportional to `cos(wₖ(a+b-c))` — a cosine wave over the output dimension `c` that peaks when `c = (a+b) mod p`.

The Fourier norms of `W_logit` peak at the same key frequencies — confirming the unembedding maps neuron outputs back to cosine waves at those frequencies.

Next, we verify the **modular structure**: neurons that respond to frequency k should only contribute to frequency-k components in the logits.

In [ ]:
# --- Recompute W_logit: MLP Output → Logits ---
# W_out [512, 128] @ W_U [128, 113] → [512, 113]
# Each row maps one MLP neuron's activation to all 113 output logits
W_logit = model.blocks[0].mlp.W_out @ model.unembed.W_U
print("W_logit", W_logit.shape)  # [512, 113]

In [ ]:
# --- W_logit in the Fourier Basis ---
# W_logit @ fourier_basis.T: [512, 113] @ [113, 113] → [512, 113]
# Each column = all 512 neurons' contributions to one Fourier component of the output
# .norm(dim=0): L2 norm across neurons for each Fourier component → [113]
# Should peak at the same 5 key frequencies — confirming the unembedding outputs only these frequencies
line((W_logit @ fourier_basis.T).norm(dim=0), x=fourier_basis_names, title="W_logit in the Fourier Basis")

> **📋 Diff from original:** Major refactor here. The original hardcodes frequency 17 with specific variable names:
> - `neurons_17 = neuron_freq_norm[17-1] > 0.85`
> - `neurons_sin_17 = neuron_acts_centered @ ...`
> - `inputs_sin_17c = fourier_basis[33] @ ...`
>
> This version introduces `study_freq = key_freqs[0]` and uses generic names: `neurons_freq`, `neurons_sin_freq`, `inputs_sin_freq` — making the code work for any key frequency.


In [ ]:
# --- Identify Neurons Belonging to Frequency 9 ---
study_freq = key_freqs[0]  # = 9 (we'll study this frequency in detail)
# Select neurons where frequency 9 explains more than 85% of their variance
neurons_freq = neuron_freq_norm[study_freq-1] > 0.85  # Boolean mask: [512]
neurons_freq.shape  # [512] — True for freq-9 neurons, False otherwise

In [ ]:
# --- Count How Many Neurons Are Dedicated to Frequency 9 ---
neurons_freq.sum()  # e.g., ~100 neurons (exact count depends on training run)

In [ ]:
# --- W_logit for Freq-9 Neurons Only, in Fourier Basis ---
# W_logit[neurons_freq]: select only freq-9 neuron rows → [n_freq9_neurons, 113]
# @ fourier_basis.T → [n_freq9_neurons, 113]
# .norm(dim=0) → [113]: total contribution per Fourier component
# Should show energy concentrated ONLY at frequency 9 — confirming modular structure
line((W_logit[neurons_freq] @ fourier_basis.T).norm(dim=0), x=fourier_basis_names, title=f"W_logit for freq {study_freq} neurons in the Fourier Basis")

### Verifying the sin(wₖ(a+b)) → sin(wₖc) pathway

We isolate the sin component at one key frequency. `neurons_sin_freq` extracts each neuron's weight for the `sin(wₖ·c)` output logit direction. Then `inputs_sin_freq = neuron_acts @ neurons_sin_freq` computes the total `sin(wₖ·c)` logit contribution as a function of (a,b).

The 2D Fourier transform should show this is dominated by `sin(wₖ(a+b))` — confirming the complete pipeline: **embed → attend → MLP computes sin/cos(wₖ(a+b)) → unembed maps to sin/cos(wₖc) logit components → together these form cos(wₖ(a+b-c))**.

In [ ]:
# --- Extract sin(w_9·c) Component of W_logit ---
# W_logit_fourier: [512, 113] — each neuron's Fourier-decomposed output profile
W_logit_fourier = W_logit @ fourier_basis  # [512, 113] @ [113, 113] → [512, 113]

# neurons_sin_freq: each neuron's weight for the sin(9·c) output direction
# Index 2*9-1 = 17 is the sin(9) component in the Fourier basis
neurons_sin_freq = W_logit_fourier[:, 2*study_freq-1]  # [512]

# Plot: most neurons have weight ~0; only freq-9 neurons have large sin(9·c) weights
line(neurons_sin_freq)

In [ ]:
# --- Check neuron_acts shape ---
neuron_acts.shape  # [12769, 512] — post-ReLU activations at '=' position for all inputs

In [ ]:
# --- Verify the Complete sin(w_k(a+b)) → sin(w_k·c) Pipeline ---
# inputs_sin_freq: weighted sum of all neuron activations using sin(9·c) weights → [12769]
# This computes the total sin(9·c) logit contribution as a function of (a, b)
inputs_sin_freq = neuron_acts @ neurons_sin_freq  # [12769, 512] @ [512] → [12769]

# 2D Fourier transform of this function of (a, b)
# If the pipeline works correctly, energy should be concentrated at sin(9) and cos(9) components
# — confirming the neurons compute sin(w_9(a+b)) which the unembedding maps to sin(w_9·c)
imshow(fourier_basis @ inputs_sin_freq.reshape(p, p) @ fourier_basis.T, title=f"Fourier Heatmap over inputs for sin {study_freq} component", x=fourier_basis_names, y=fourier_basis_names)

> **📋 Diff from original:** The original section header is `# Black Box Methods + Progress Measures`. Renamed to `# Progress Measures — Grokking is Gradual, Not Sudden`.


# Progress Measures — Grokking is Gradual, Not Sudden

The analysis above shows that the trained model uses a Fourier-based algorithm. But **when** does this algorithm form during training? Train and test loss suggest a sudden transition, but progress measures reveal the truth: the Fourier circuit forms **gradually** during training, long before grokking occurs.

The paper identifies **three phases** (epoch ranges below are from this notebook's training run; the paper's mainline model uses different boundaries: ~1.4k, ~9.4k, ~14k):
1. **Memorization** (epochs 0–1,500): Model memorizes training data with unstructured weights
2. **Circuit formation** (epochs 1,500–13,300): Fourier circuit gradually builds up in the weights, but memorization still dominates
3. **Cleanup** (epochs 13,300–16,600): Weight decay removes memorization components; the Fourier circuit becomes dominant → test loss drops (grokking!)

## Setup Code for Progress Measures

Helper to plot embedding Fourier norms as bar charts, splitting sin and cos components for each frequency `wₖ`:

In [ ]:
# --- Helper: Split Fourier Embedding into Sin and Cos Components ---
def embed_to_cos_sin(fourier_embed):
    """Separate Fourier embedding into sin (odd indices) and cos (even indices) components."""
    if len(fourier_embed.shape) == 1:
        # 1D input: fourier_embed[1::2] = sin components, [2::2] = cos components
        return torch.stack([fourier_embed[1::2], fourier_embed[2::2]])  # [2, n_freqs]
    else:
        # 2D input: same split along dim=1
        return torch.stack([fourier_embed[:, 1::2], fourier_embed[:, 2::2]], dim=1)  # [batch, 2, n_freqs]

from neel_plotly.plot import melt  # Helper to convert tensors to "melted" DataFrames for Plotly

def plot_embed_bars(
    fourier_embed,
    title="Norm of embedding of each Fourier Component",
    return_fig=False,
    **kwargs
):
    """Bar chart showing sin and cos norms at each frequency."""
    cos_sin_embed = embed_to_cos_sin(fourier_embed)  # Split into sin/cos
    df = melt(cos_sin_embed)                          # Convert to DataFrame with columns: 0, 1, value
    group_labels = {0: "sin", 1: "cos"}
    df["Trig"] = df["0"].map(lambda x: group_labels[x])  # Label each row as "sin" or "cos"
    fig = px.bar(
        df,
        barmode="group",       # Side-by-side bars (not stacked)
        color="Trig",          # Color by sin/cos
        x="1",                 # X-axis = frequency index
        y="value",             # Y-axis = norm value
        labels={"1": "$w_k$", "value": "Norm"},
        title=title,
        **kwargs
    )
    fig.update_layout(dict(legend_title=""))

    if return_fig:
        return fig
    else:
        fig.show()

Helper to evaluate cross-entropy loss on a tensor of logits. Supports computing loss on train, test, or all data. The `bias_correction` option adjusts for missing bias terms by centering the new logits and adding back the original mean — useful when testing modified logits that may have shifted baseline.

In [ ]:
# --- Helper: Evaluate Loss on Custom Logits ---
def test_logits(logits, bias_correction=False, original_logits=None, mode="all"):
    """Compute cross-entropy loss for logits representing all p^2 inputs.
    Supports train/test/all splits and optional bias correction."""

    # Handle various input shapes
    if logits.shape[1] == p * p:
        logits = logits.T                  # If transposed (features × batch), fix it
    if logits.shape == torch.Size([p * p, p + 1]):
        logits = logits[:, :-1]            # Remove '=' token column if present
    logits = logits.reshape(p * p, p)      # Ensure shape [12769, 113]

    if bias_correction:
        # Bias correction: when testing modified logits that may have a shifted baseline,
        # center the new logits along the batch dimension and add back the original logit mean.
        # This accounts for missing bias terms.
        logits = (
            einops.reduce(original_logits - logits, "batch ... -> ...", "mean") + logits
        )

    # Evaluate loss on the specified split
    if mode == "train":
        return loss_fn(logits[train_indices], labels[train_indices])
    elif mode == "test":
        return loss_fn(logits[test_indices], labels[test_indices])
    elif mode == "all":
        return loss_fn(logits, labels)

Framework for computing any metric across all 250 training checkpoints. `get_metrics` loads each saved checkpoint into the model, runs `metric_fn`, and collects results. This lets us track how any property (loss, Fourier coefficients, etc.) evolves over training.

In [ ]:
# --- Initialize Metric Cache ---
# Dictionary that will store arrays of metric values across all 250 training checkpoints.
# Keys are metric names (e.g., "cos_coeffs", "restricted_loss"), values are tensors.
metric_cache = {}

In [ ]:
# --- Framework for Computing Metrics Across All Checkpoints ---
def get_metrics(model, metric_cache, metric_fn, name, reset=False):
    """Load each of the 250 saved checkpoints, compute metric_fn(model), and store results."""
    if reset or (name not in metric_cache) or (len(metric_cache[name]) == 0):
        metric_cache[name] = []
        for c, sd in enumerate(tqdm.tqdm((model_checkpoints))):  # Iterate over 250 checkpoints
            model.reset_hooks()          # Clear any lingering hooks from previous iterations
            model.load_state_dict(sd)    # Load this checkpoint's weights into the model
            out = metric_fn(model)       # Compute the metric using the current model state
            if type(out) == torch.Tensor:
                out = utils.to_numpy(out)  # Convert tensor to numpy for storage
            metric_cache[name].append(out)
        model.load_state_dict(model_checkpoints[-1])  # IMPORTANT: restore the final trained model
        try:
            metric_cache[name] = torch.tensor(metric_cache[name])  # Convert list to tensor
        except:
            metric_cache[name] = torch.tensor(np.array(metric_cache[name]))  # Fallback for mixed types

## Defining Progress Measures

Progress measures are metrics that change **continuously** during training, revealing the gradual structure formation that underlies the seemingly **discontinuous** jump in test accuracy (grokking). They let us "see through" the flat test loss curve to the circuit being built underneath.

### Loss Curves with Phase Boundaries

We mark the three training phases with dashed vertical lines:
- **Memorization end** (~1,500): Train loss has reached near-zero
- **Circuit formation end** (~13,300): The Fourier circuit is fully formed; test loss is about to drop
- **Cleanup end** (~16,600): Memorization components removed; test loss has fully converged

In [ ]:
# --- Phase Boundary Epochs ---
# Approximate epochs where each training phase ends (based on loss curve observations):
memorization_end_epoch = 1500      # Phase 1 ends: training data is memorized
circuit_formation_end_epoch = 13300 # Phase 2 ends: Fourier circuit is fully formed
cleanup_end_epoch = 16600          # Phase 3 ends: memorization removed, test loss converged

In [ ]:
# --- Helper: Add Phase Boundary Lines to Plots ---
def add_lines(figure):
    """Add 3 dashed vertical lines marking the phase transitions."""
    figure.add_vline(memorization_end_epoch, line_dash="dash", opacity=0.7)       # End of memorization
    figure.add_vline(circuit_formation_end_epoch, line_dash="dash", opacity=0.7)  # End of circuit formation
    figure.add_vline(cleanup_end_epoch, line_dash="dash", opacity=0.7)            # End of cleanup
    return figure

In [ ]:
# --- Training Curves with Phase Boundary Lines ---
# Same plot as cell 40, but now overlaid with dashed lines marking the 3 training phases.
fig = line([train_losses[::100], test_losses[::100]], x=np.arange(0, len(train_losses), 100), xaxis="Epoch", yaxis="Loss", log_y=True, title="Training Curve for Modular Addition", line_labels=['train', 'test'], toggle_x=True, toggle_y=True, return_fig=True)
add_lines(fig)  # Add vertical dashed lines at epochs 1500, 13300, 16600

### Logit Periodicity — The cos(wₖ(a+b-c)) Structure

The paper's key prediction: the output logits should be well-approximated by a **sum of cosine waves**:

$$\text{logit}(a, b, c) \approx \sum_{k \in \text{key\_freqs}} \alpha_k \cdot \cos\!\left(\frac{2\pi k}{p}(a + b - c)\right)$$

This function is maximized when `c = (a+b) mod p` because all cosines equal 1 at argument 0 (constructive interference), while at other values of c the waves cancel out (destructive interference).

Below, we construct the predicted `cos(wₖ(a+b-c))` tensor for each key frequency and measure how well it matches the actual logits.

In [ ]:
# --- Reshape Logits to 3D Tensor (a, b, c) ---
# Take logits at the '=' position only: [12769, 113]
all_logits = original_logits[:, -1, :]
print(all_logits.shape)  # [12769, 113]

# Reshape to [113, 113, 113] indexed by (a, b, c) where c is the predicted output class.
# This 3D "logit cube" shows the model's confidence for every (a, b, c) triple.
all_logits = einops.rearrange(all_logits, "(a b) c -> a b c", a=p, b=p)
print(all_logits.shape)  # [113, 113, 113]

In [ ]:
# --- Build Theoretical cos(w_k(a+b-c)) Cubes for Key Frequencies ---
# The paper predicts: logit(a, b, c) ≈ Σ_k α_k · cos(w_k(a+b-c))
# which peaks at c = (a+b) mod p via constructive interference.
# Here we build the unit-normalized cos(w_k(a+b-c)) tensor for each key frequency.
coses = {}
for freq in key_freqs:  # [9, 33, 36, 38, 55]
    print("Freq:", freq)
    a = torch.arange(p)[:, None, None]  # [113, 1, 1] — broadcast over b and c
    b = torch.arange(p)[None, :, None]  # [1, 113, 1] — broadcast over a and c
    c = torch.arange(p)[None, None, :]  # [1, 1, 113] — broadcast over a and b
    # cos(2π·freq·(a+b-c)/113) for all (a, b, c) combinations → [113, 113, 113]
    cube_predicted_logits = torch.cos(freq * 2 * torch.pi / p * (a + b - c)).to(device)
    cube_predicted_logits /= cube_predicted_logits.norm()  # Normalize to unit norm
    coses[freq] = cube_predicted_logits

In [ ]:
# --- Project Actual Logits onto the 5 Cosine Cubes ---
# Decompose the logit cube into a sum of cos(w_k(a+b-c)) components.
approximated_logits = torch.zeros_like(all_logits)  # [113, 113, 113] of zeros
for freq in key_freqs:
    print("Freq:", freq)
    # Dot product with unit-norm cosine cube = projection coefficient (scalar)
    coeff = (all_logits * coses[freq]).sum()
    print("Coeff:", coeff)
    # Cosine similarity = coeff / ||logits|| (since coses are already unit norm)
    cosine_sim = coeff / all_logits.norm()
    print("Cosine Sim:", cosine_sim)
    # Accumulate: approximation += coefficient × cosine_cube
    approximated_logits += coeff * coses[freq]

# Measure the residual (everything NOT explained by the 5 key frequencies)
residual = all_logits - approximated_logits
print("Residual size:", residual.norm())
print("Residual fraction of norm:", residual.norm()/all_logits.norm())
# Expected: residual is ~2% of total norm — the 5 frequencies explain ~98%

In [ ]:
# --- Control: Cosine Similarity with a Random Direction ---
# A random unit vector in the same space should have near-zero cosine similarity with the logits.
# This confirms the cosine structure found above is real, not a statistical artifact.
random_logit_cube = torch.randn_like(all_logits)
print((all_logits * random_logit_cube).sum()/random_logit_cube.norm()/all_logits.norm())
# Expected: ~0.001 (basically zero)

In [ ]:
# --- Loss of the Full Model ---
test_logits(all_logits)  # ~1.16e-6

In [ ]:
# --- Loss of the 5-Frequency Approximation ---
# Using ONLY the 5 cosine cubes actually gives BETTER loss than the full model!
# The residual is noise that slightly HURTS performance.
test_logits(approximated_logits)  # ~3.25e-8 (better than original!)

**Results**: The 5 key frequencies together explain ~98% of the logit norm. The approximated logits (using only the 5 cosine cubes) actually achieve **better** loss than the original (3.25e-8 vs 1.16e-6) — the residual is noise that slightly *hurts* performance. A random direction has near-zero cosine similarity (~0.001), confirming this structure is real and not a statistical artifact.

#### Tracking Fourier Coefficients During Training

We build `cos(wₖ(a+b-c))` tensors for **all 56** possible frequencies (not just the 5 key ones), then track how strongly the model's logits project onto each one throughout training. This reveals:
- During **memorization**: all coefficients are small (the model uses unstructured memorization, not Fourier modes)
- During **circuit formation**: key frequency coefficients gradually grow (the Fourier circuit is being built)
- During **cleanup**: key frequency coefficients stabilize, non-key ones shrink further

In [ ]:
# --- Build Cosine Cubes for ALL 56 Frequencies (Not Just Key Ones) ---
# Used to track how every possible frequency's coefficient evolves during training.
cos_cube = []
for freq in range(1, p//2 + 1):  # freq = 1, 2, ..., 56
    a = torch.arange(p)[:, None, None]  # [113, 1, 1]
    b = torch.arange(p)[None, :, None]  # [1, 113, 1]
    c = torch.arange(p)[None, None, :]  # [1, 1, 113]
    cube_predicted_logits = torch.cos(freq * 2 * torch.pi / p * (a + b - c)).to(device)  # [113, 113, 113]
    cube_predicted_logits /= cube_predicted_logits.norm()  # Unit normalize
    cos_cube.append(cube_predicted_logits)
cos_cube = torch.stack(cos_cube, dim=0)  # [56, 113, 113, 113]
print(cos_cube.shape)

In [ ]:
# --- Metric: Fourier Projection Coefficients at Each Checkpoint ---
def get_cos_coeffs(model):
    """Compute how much the model's logits project onto each of the 56 cos(w_k(a+b-c)) cubes."""
    logits = model(dataset)[:, -1]  # Run model, take logits at '=' position → [12769, 113]
    logits = einops.rearrange(logits, "(a b) c -> a b c", a=p, b=p)  # → [113, 113, 113]
    # Element-wise multiply with all 56 cosine cubes [56, 113, 113, 113] and sum → [56]
    vals = (cos_cube * logits[None, :, :, :]).sum([-3, -2, -1])
    return vals  # Projection coefficient for each frequency

# Run across all 250 checkpoints
get_metrics(model, metric_cache, get_cos_coeffs, "cos_coeffs")
print(metric_cache["cos_coeffs"].shape)  # [250, 56] — 250 checkpoints × 56 frequencies

In [ ]:
# --- Plot Fourier Coefficients Over Training ---
# .T transposes to [56, 250] — one line per frequency.
# Key frequencies grow during circuit formation; non-key frequencies stay near 0.
fig = line(metric_cache["cos_coeffs"].T, line_labels=[f"Freq {i}" for i in range(1, p//2+1)], title="Coefficients with Predicted Logits", xaxis="Epoch", x=checkpoint_epochs, yaxis="Coefficient", return_fig=True)
add_lines(fig)

In [ ]:
# --- Metric: Cosine Similarity of Logits with Each Frequency ---
def get_cos_sim(model):
    """Same as get_cos_coeffs but normalized by logit norm → scale-invariant cosine similarity."""
    logits = model(dataset)[:, -1]
    logits = einops.rearrange(logits, "(a b) c -> a b c", a=p, b=p)
    vals = (cos_cube * logits[None, :, :, :]).sum([-3, -2, -1])
    return vals / logits.norm()  # Divide by ||logits|| to get cosine similarity

get_metrics(model, metric_cache, get_cos_sim, "cos_sim")
print(metric_cache["cos_sim"].shape)  # [250, 56]

# Plot: cosine similarity shows how much of the logit DIRECTION aligns with each frequency
fig = line(metric_cache["cos_sim"].T, line_labels=[f"Freq {i}" for i in range(1, p//2+1)], title="Cosine Sim with Predicted Logits", xaxis="Epoch", x=checkpoint_epochs, yaxis="Cosine Sim", return_fig=True)
add_lines(fig)

In [ ]:
# --- Metric: Residual Fraction After Projecting Out All Frequencies ---
def get_residual_cos_sim(model):
    """Compute what fraction of logit norm is NOT explained by any cos(w_k(a+b-c))."""
    logits = model(dataset)[:, -1]
    logits = einops.rearrange(logits, "(a b) c -> a b c", a=p, b=p)
    vals = (cos_cube * logits[None, :, :, :]).sum([-3, -2, -1])  # [56] projection coefficients
    # Reconstruct approximation and compute residual
    residual = logits - (vals[:, None, None, None] * cos_cube).sum(dim=0)
    return residual.norm() / logits.norm()  # Fraction of norm that's unexplained

get_metrics(model, metric_cache, get_residual_cos_sim, "residual_cos_sim")
print(metric_cache["residual_cos_sim"].shape)  # [250]

# Plot all 56 cosine similarities + residual fraction, showing the gradual Fourier transition
fig = line([metric_cache["cos_sim"][:, i] for i in range(p//2)]+[metric_cache["residual_cos_sim"]], line_labels=[f"Freq {i}" for i in range(1, p//2+1)]+["residual"], title="Cosine Sim with Predicted Logits + Residual", xaxis="Epoch", x=checkpoint_epochs, yaxis="Cosine Sim", return_fig=True)
add_lines(fig)

## Restricted Loss — Measuring the Generalizing Circuit

**Restricted loss** keeps **only** the key frequency Fourier components of the neuron activations and discards everything else. Specifically, for each key frequency wₖ, we project each neuron's (a,b)-dependent activation onto `cos(wₖ(a+b))` and `sin(wₖ(a+b))`, keeping only these 10 directions (2 per frequency × 5 frequencies) plus the mean.

If the Fourier circuit is sufficient for generalization, the restricted logits should achieve low loss even though most of the neuron activations have been discarded. The restricted loss acts as a **progress measure** — it decreases during circuit formation, revealing that the generalizing mechanism is being built even while test loss appears unchanged.

In [ ]:
# --- Check neuron_acts shape ---
neuron_acts.shape  # [12769, 512] — post-ReLU MLP activations at '=' position

In [ ]:
# --- 2D Fourier Transform of Centered Neuron Activations ---
# Reshape flat batch [12769, 512] to grid [113, 113, 512] indexed by (a, b, neuron)
neuron_acts_square = einops.rearrange(neuron_acts, "(a b) neur -> a b neur", a=p, b=p).clone()

# Center each neuron by subtracting its mean activation across all (a, b) inputs.
# "a b neur -> 1 1 neur" computes mean over (a, b) dims → [1, 1, 512]
neuron_acts_square -= einops.reduce(neuron_acts_square, "a b neur -> 1 1 neur", "mean")

# 2D DFT using Einstein summation: project both a and b axes into the Fourier basis.
# "a b neur, fa a, fb b -> fa fb neur" contracts both spatial dims → [113, 113, 512]
neuron_acts_square_fourier = einsum("a b neur, fa a, fb b -> fa fb neur", neuron_acts_square, fourier_basis, fourier_basis)

# Plot norm across neurons: shows which 2D Fourier components have energy across the whole network
imshow(neuron_acts_square_fourier.norm(dim=-1), xaxis="Fourier Component b", yaxis="Fourier Component a", title="Norms of neuron activations by Fourier Component", x=fourier_basis_names, y=fourier_basis_names)

In [ ]:
# --- Refresh Model Activations ---
# Re-run the model (in case get_metrics loaded a different checkpoint) and re-extract activations.
original_logits, cache = model.run_with_cache(dataset)
print(original_logits.numel())  # 4,328,721
neuron_acts = cache["post", 0, "mlp"][:, -1, :]  # Refresh neuron_acts: [12769, 512]

In [ ]:
# --- Compute Restricted Logits (Keep Only Key Frequency Components) ---
# Start with zeros and add back only the key-frequency components of the neuron activations.
approx_neuron_acts = torch.zeros_like(neuron_acts)     # [12769, 512] of zeros
approx_neuron_acts += neuron_acts.mean(dim=0)           # Add the mean activation per neuron (bias term)

a = torch.arange(p)[:, None]  # [113, 1] — for broadcasting over (a, b) grid
b = torch.arange(p)[None, :]  # [1, 113]

for freq in key_freqs:  # [9, 33, 36, 38, 55]
    # --- Project onto cos(w_k(a+b)) ---
    cos_apb_vec = torch.cos(freq * 2 * torch.pi / p * (a + b)).to(device)  # [113, 113]
    cos_apb_vec /= cos_apb_vec.norm()                     # Normalize to unit vector
    cos_apb_vec = einops.rearrange(cos_apb_vec, "a b -> (a b) 1")  # [12769, 1]
    # Project: coefficient = sum(neuron_acts * cos_vec) along batch dim → [512]
    # Reconstruct: coefficient * cos_vec → adds back only the cos(w_k(a+b)) component
    approx_neuron_acts += (neuron_acts * cos_apb_vec).sum(dim=0) * cos_apb_vec

    # --- Project onto sin(w_k(a+b)) ---
    sin_apb_vec = torch.sin(freq * 2 * torch.pi / p * (a + b)).to(device)  # [113, 113]
    sin_apb_vec /= sin_apb_vec.norm()
    sin_apb_vec = einops.rearrange(sin_apb_vec, "a b -> (a b) 1")  # [12769, 1]
    approx_neuron_acts += (neuron_acts * sin_apb_vec).sum(dim=0) * sin_apb_vec

# Compute logits from restricted neuron activations: [12769, 512] @ [512, 113] → [12769, 113]
restricted_logits = approx_neuron_acts @ W_logit

# Test loss on the TEST set — should be very low (~9e-8), even better than full model!
print(loss_fn(restricted_logits[test_indices], test_labels))

> **📋 Diff from original:** The original has `print(loss_fn(all_logits, labels))` with a comment `# This bugged on models not fully trained`. This version fixes the bug by reshaping: `loss_fn(all_logits.reshape(-1, all_logits.shape[-1]), labels)`. `CrossEntropyLoss` expects shape `(N, C)` not `(batch, seq, C)`.


In [ ]:
# --- Compare: Full Model Loss ---
# The full model's loss is slightly WORSE than restricted — the non-key components are noise.
print(loss_fn(all_logits.reshape(-1, all_logits.shape[-1]), labels))  # ~1.16e-6

**Result**: Restricted loss ~9e-8 — even **better** than the full model's loss of ~1.2e-6. Discarding the non-key-frequency components *improves* performance. This proves:
1. The key frequencies contain **all** the useful generalizing information
2. The remaining components are memorization noise that slightly hurts generalization

### Restricted Loss During Training

Track restricted loss across all checkpoints. Key observation: **restricted loss starts decreasing during circuit formation** (around epoch 1,500–13,000), long before test loss drops. This proves the Fourier circuit is being assembled gradually — grokking is not sudden!

The ratio `test_loss / restricted_loss` grows during circuit formation: the circuit is ready, but memorization noise is masking it.

In [ ]:
# --- Restricted Loss as a Metric Function (for use with get_metrics) ---
def get_restricted_loss(model):
    """Keep only key-frequency neuron components, compute test loss.
    This measures how well the generalizing Fourier circuit alone performs."""
    logits, cache = model.run_with_cache(dataset)
    logits = logits[:, -1, :]  # [12769, 113] — logits at '=' position
    neuron_acts = cache["post", 0, "mlp"][:, -1, :]  # [12769, 512]

    # Start with mean activation, then add key-frequency projections
    approx_neuron_acts = torch.zeros_like(neuron_acts)
    approx_neuron_acts += neuron_acts.mean(dim=0)
    a = torch.arange(p)[:, None]
    b = torch.arange(p)[None, :]
    for freq in key_freqs:
        cos_apb_vec = torch.cos(freq * 2 * torch.pi / p * (a + b)).to(device)
        cos_apb_vec /= cos_apb_vec.norm()
        cos_apb_vec = einops.rearrange(cos_apb_vec, "a b -> (a b) 1")
        approx_neuron_acts += (neuron_acts * cos_apb_vec).sum(dim=0) * cos_apb_vec
        sin_apb_vec = torch.sin(freq * 2 * torch.pi / p * (a + b)).to(device)
        sin_apb_vec /= sin_apb_vec.norm()
        sin_apb_vec = einops.rearrange(sin_apb_vec, "a b -> (a b) 1")
        approx_neuron_acts += (neuron_acts * sin_apb_vec).sum(dim=0) * sin_apb_vec

    # Compute restricted logits: restricted_neuron_acts → W_out → W_U → logits
    restricted_logits = approx_neuron_acts @ model.blocks[0].mlp.W_out @ model.unembed.W_U
    # Bias correction: center restricted logits to match original logit mean
    restricted_logits += logits.mean(dim=0, keepdim=True) - restricted_logits.mean(dim=0, keepdim=True)
    return loss_fn(restricted_logits[test_indices], test_labels)  # Evaluate on TEST set

get_restricted_loss(model)

In [ ]:
# --- Compute Restricted Loss Across All 250 Checkpoints ---
get_metrics(model, metric_cache, get_restricted_loss, "restricted_loss", reset=True)
print(metric_cache["restricted_loss"].shape)  # [250] — one loss value per checkpoint

In [ ]:
# --- Plot: Restricted Loss vs Train/Test Loss ---
# Three curves: train loss, test loss, and restricted loss.
# Key insight: restricted loss drops GRADUALLY during circuit formation (epochs 1500-13300),
# revealing that the Fourier circuit is being built even while test loss appears unchanged.
fig = line([train_losses[::100], test_losses[::100], metric_cache["restricted_loss"]], x=np.arange(0, len(train_losses), 100), xaxis="Epoch", yaxis="Loss", log_y=True, title="Restricted Loss Curve", line_labels=['train', 'test', "restricted_loss"], toggle_x=True, toggle_y=True, return_fig=True)
add_lines(fig)

In [ ]:
# --- Plot: Test Loss / Restricted Loss Ratio ---
# This ratio grows during circuit formation: the Fourier circuit is ready (low restricted loss),
# but memorization noise in the full model masks it (high test loss).
# The growing gap shows how much the memorization component is hurting generalization.
fig = line([torch.tensor(test_losses[::100])/metric_cache["restricted_loss"]], x=np.arange(0, len(train_losses), 100), xaxis="Epoch", yaxis="Loss", log_y=True, title="Restricted Loss to Test Loss Ratio", toggle_x=True, toggle_y=True, return_fig=True)
# WARNING: bugged when cancelling training half way through
add_lines(fig)

## Excluded Loss — Measuring the Memorization Component

**Excluded loss** is the complement of restricted loss — it **removes** the key frequency components and keeps everything else. We then compute cross-entropy loss on the **training set** using only these non-Fourier components.

- If the model is **memorizing**, the excluded (non-Fourier) components carry useful information → excluded loss is low
- If the model has **cleaned up** memorization, the non-Fourier components are just noise → excluded loss is high (near random chance ~ln(113))

This metric tracks the **death of memorization**: excluded loss rises during the cleanup phase as weight decay strips away the non-generalizing components.

In [ ]:
# --- Compute Excluded Logits (Remove Key Frequency Components, Keep Everything Else) ---
# This is the COMPLEMENT of restricted loss — it measures the memorization component.
approx_neuron_acts = torch.zeros_like(neuron_acts)  # [12769, 512]
# Note: does NOT add the mean (unlike restricted loss)
a = torch.arange(p)[:, None]
b = torch.arange(p)[None, :]

# Project onto key-frequency components (same as restricted, but we'll SUBTRACT them)
for freq in key_freqs:
    cos_apb_vec = torch.cos(freq * 2 * torch.pi / p * (a + b)).to(device)
    cos_apb_vec /= cos_apb_vec.norm()
    cos_apb_vec = einops.rearrange(cos_apb_vec, "a b -> (a b) 1")
    approx_neuron_acts += (neuron_acts * cos_apb_vec).sum(dim=0) * cos_apb_vec
    sin_apb_vec = torch.sin(freq * 2 * torch.pi / p * (a + b)).to(device)
    sin_apb_vec /= sin_apb_vec.norm()
    sin_apb_vec = einops.rearrange(sin_apb_vec, "a b -> (a b) 1")
    approx_neuron_acts += (neuron_acts * sin_apb_vec).sum(dim=0) * sin_apb_vec

# SUBTRACT key-frequency components → keep only non-Fourier (memorization) components
excluded_neuron_acts = neuron_acts - approx_neuron_acts
excluded_logits = excluded_neuron_acts @ W_logit  # [12769, 113]

# Test on TRAINING set: if memorization exists, this should have low loss
print(loss_fn(excluded_logits[train_indices], train_labels))

In [ ]:
# --- Excluded Loss as a Metric Function (for use with get_metrics) ---
def get_excluded_loss(model):
    """Remove key-frequency components, compute TRAIN loss on the remainder.
    Measures the strength of the memorization component."""
    logits, cache = model.run_with_cache(dataset)
    logits = logits[:, -1, :]
    neuron_acts = cache["post", 0, "mlp"][:, -1, :]

    # Project onto key-frequency components
    approx_neuron_acts = torch.zeros_like(neuron_acts)
    # Note: no mean added (unlike restricted loss)
    a = torch.arange(p)[:, None]
    b = torch.arange(p)[None, :]
    for freq in key_freqs:
        cos_apb_vec = torch.cos(freq * 2 * torch.pi / p * (a + b)).to(device)
        cos_apb_vec /= cos_apb_vec.norm()
        cos_apb_vec = einops.rearrange(cos_apb_vec, "a b -> (a b) 1")
        approx_neuron_acts += (neuron_acts * cos_apb_vec).sum(dim=0) * cos_apb_vec
        sin_apb_vec = torch.sin(freq * 2 * torch.pi / p * (a + b)).to(device)
        sin_apb_vec /= sin_apb_vec.norm()
        sin_apb_vec = einops.rearrange(sin_apb_vec, "a b -> (a b) 1")
        approx_neuron_acts += (neuron_acts * sin_apb_vec).sum(dim=0) * sin_apb_vec

    # Subtract key-frequency components from neuron activations
    excluded_neuron_acts = neuron_acts - approx_neuron_acts
    # Reconstruct the full residual stream: excluded MLP output + residual stream before MLP
    # cache["resid_mid", 0][:, -1, :] = residual stream AFTER attention, BEFORE MLP
    residual_stream_final = excluded_neuron_acts @ model.blocks[0].mlp.W_out + cache["resid_mid", 0][:, -1, :]
    excluded_logits = residual_stream_final @ model.unembed.W_U  # → [12769, 113]
    return loss_fn(excluded_logits[train_indices], train_labels)  # Evaluate on TRAIN set

get_excluded_loss(model)

In [ ]:
# --- Compute Excluded Loss Across All 250 Checkpoints ---
get_metrics(model, metric_cache, get_excluded_loss, "excluded_loss", reset=True)
print(metric_cache["excluded_loss"].shape)  # [250]

In [ ]:
# --- Final Combined Plot: All 4 Loss Curves ---
# This is the definitive visualization showing all progress measures together:
#   - train_loss: drops early (memorization achieved)
#   - test_loss: flat plateau → sudden drop (grokking!)
#   - restricted_loss: GRADUALLY decreases (Fourier circuit being built)
#   - excluded_loss: stays low, then RISES (memorization being eroded by weight decay)
fig = line([train_losses[::100], test_losses[::100], metric_cache["excluded_loss"], metric_cache["restricted_loss"]], x=np.arange(0, len(train_losses), 100), xaxis="Epoch", yaxis="Loss", log_y=True, title="Excluded and Restricted Loss Curve", line_labels=['train', 'test', "excluded_loss", "restricted_loss"], toggle_x=True, toggle_y=True, return_fig=True)

add_lines(fig)

> **📋 Diff from original:** This Summary cell does not exist in the original notebook. It was added to synthesize the key findings.


# Summary

This notebook has demonstrated the complete mechanistic interpretability analysis of grokking:

**The Algorithm:**
1. The embedding `W_E` maps each input token `a` to Fourier components: `sin(wₖa)`, `cos(wₖa)` at 5 key frequencies
2. Attention heads copy these Fourier-encoded embeddings of `a` and `b` to the `=` position
3. MLP neurons compute `cos(wₖ(a+b))` and `sin(wₖ(a+b))` using the trig identity `cos(α+β) = cos(α)cos(β) − sin(α)sin(β)`
4. The unembedding `W_U` produces logits proportional to `Σₖ cos(wₖ(a+b−c))`, which peak at the correct answer `c = (a+b) mod 113` via constructive interference

**The Three Phases of Training:**
1. **Memorization** (0–1.5k epochs): Model memorizes training data; excluded loss is low (memorization works on training set) but restricted loss remains high (Fourier circuit not yet formed)
2. **Circuit formation** (1.5k–13k epochs): Fourier circuit gradually builds (restricted loss drops); memorization persists (excluded loss stays low); test loss appears unchanged
3. **Cleanup** (13k–17k epochs): Weight decay erodes memorization (excluded loss rises); Fourier circuit becomes dominant; test loss suddenly drops — **this is grokking**

**Key Insight**: Grokking is not a sudden phase transition. It is the **delayed visibility** of a gradually forming algorithm, caused by the slow removal of a competing memorization solution under weight decay pressure.